# ReStructure Data JSON
---

In [11]:
import re
import json
import csv
import argparse
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# กำหนด Path ตามโจทย์
INPUT_DIR = Path('data')
OUTPUT_DIR = Path('struc-data')

# สร้างโฟลเดอร์ output ถ้ายังไม่มี
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Input directory: data
Output directory: struc-data


## Text Cleaning
---

In [12]:
# Patterns สำหรับลบ header/footer ของราชกิจจานุเบกษา
_HEADER_PATTERNS = [
    re.compile(r'วัน(?:ที่|พุธที่|จันทร์ที่|อังคารที่|พฤหัสที่|ศุกร์ที่|เสาร์ที่|อาทิตย์ที่)?\s*[\d๐-๙]+\s*\S+\s*[\d๐-๙]+\s*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'เล่ม\s*[\d๐-๙]+\s*หน้า\s*[\d๐-๙]+\s*ราชกิจจานุเบกษา[^\n]*'),
    re.compile(r'^ราชกิจจานุเบกษา[^\n]*$', re.MULTILINE),
    re.compile(r'^---+$', re.MULTILINE),
    re.compile(r'^\s*[\d๐-๙]+\s*$', re.MULTILINE),
]

def _fix_sara_am(text: str) -> str:
    """แก้ bug สระอำแตก: 'จ านวน' → 'จำนวน'"""
    return re.sub(r'([ก-ฮ])\s+า([ก-ฮ\s])', r'\1ำ\2', text)

def _thai_to_arabic(text: str) -> str:
    """แปลงเลขไทย → เลขอารบิก"""
    mapping = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    return text.translate(mapping)

def clean_page_text(raw: str) -> str:
    """Clean raw_markdown ของแต่ละหน้า"""
    text = raw
    for pat in _HEADER_PATTERNS:
        text = pat.sub('', text)
    text = _fix_sara_am(text)
    text = _thai_to_arabic(text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

## Structure Parsing
---

In [13]:
# Regex patterns
_RE_CHAPTER = re.compile(r'^(?:หมวด(?:ที่)?\s*(\d+)\s*(.*)|บท(?:ที่)?\s*(\d+)\s*(.*))$', re.MULTILINE)
_RE_SPECIAL_CHAPTER = re.compile(r'^(บททั่วไป|บทเฉพาะ(?:กาล)?|บทบัญญัติเฉพาะ(?:กาล)?|บทนำ)\s*$', re.MULTILINE)
_RE_SECTION = re.compile(r'มาตรา\s+(\d+)\s+(.*?)(?=\s*มาตรา\s+\d+|\s*หมวด|\s*บท(?:ที่|ทั่วไป|เฉพาะ)|$)', re.DOTALL)

@dataclass
class Chapter:
    chapter_number: int  # 0 = บททั่วไป, -1 = บทเฉพาะกาล
    chapter_title: str
    sections: list = field(default_factory=list)

def _parse_sections_from_text(text: str) -> list[dict]:
    sections = []
    for m in _RE_SECTION.finditer(text):
        sec_num = int(m.group(1))
        sec_text = re.sub(r'\s+', ' ', m.group(2)).strip()
        if sec_text:
            sections.append({"section_number": sec_num, "text": sec_text})
    return sections

def _parse_full_text(full_text: str) -> tuple[str, list[Chapter]]:
    preamble = ""
    first_section_match = re.search(r'มาตรา\s+1\b', full_text)
    if first_section_match:
        preamble_raw = full_text[:first_section_match.start()]
        preamble = re.sub(r'\s+', ' ', preamble_raw).strip()[:2000]

    chapters: list[Chapter] = []
    chapter_splits = []

    for m in _RE_SPECIAL_CHAPTER.finditer(full_text):
        title = m.group(1)
        chap_num = 0 if 'ทั่วไป' in title or 'นำ' in title else -1
        chapter_splits.append((m.start(), chap_num, title))

    for m in _RE_CHAPTER.finditer(full_text):
        if m.group(1): chapter_splits.append((m.start(), int(m.group(1)), m.group(2).strip()))
        elif m.group(3): chapter_splits.append((m.start(), int(m.group(3)), m.group(4).strip()))

    chapter_splits.sort(key=lambda x: x[0])

    if not chapter_splits:
        sections = _parse_sections_from_text(full_text)
        chapters.append(Chapter(chapter_number=0, chapter_title="บททั่วไป", sections=sections))
        return preamble, chapters

    for i, (pos, chap_num, chap_title) in enumerate(chapter_splits):
        end_pos = chapter_splits[i+1][0] if i+1 < len(chapter_splits) else len(full_text)
        segment = full_text[pos:end_pos]
        chap = Chapter(chapter_number=chap_num, chapter_title=chap_title)
        chap.sections = _parse_sections_from_text(segment)
        chapters.append(chap)

    return preamble, chapters

## Main Processor
---

In [ ]:
def _save_sections_csv(data: dict, csv_path: Path):
    rows = []
    for chap in data['chapters']:
        for sec in chap['sections']:
            rows.append({
                'constitution_id': data['id'],
                'year_th': data['year_th'],
                'year_ce': data['year_ce'],
                'name_short': data['name_short'],
                'era': data.get('era', ''),
                'regime_type': data.get('regime_type', ''),
                'chapter_number': chap['chapter_number'],
                'chapter_title': chap['chapter_title'],
                'section_number': sec['section_number'],
                'section_text': sec['text'],
            })
    if rows:
        with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

def process_constitution_json(input_file: Path, output_dir: Path) -> dict:
    with open(input_file, encoding='utf-8') as f:
        raw_data = json.load(f)

    year = raw_data.get('year_th', 0)
    pages = raw_data.get('pages', [])
    combined_raw = raw_data.get('full_text') or "\n\n".join(p.get('raw_markdown', '') for p in pages if p.get('raw_markdown'))
    
    clean_text = clean_page_text(combined_raw)
    preamble, chapters = _parse_full_text(clean_text)
    total_sections = sum(len(c.sections) for c in chapters)

    structured = {
        "id": raw_data.get('id', f'const_{year}'),
        "year_th": year,
        "year_ce": raw_data.get('year_ce', year - 543),
        "name_short": raw_data.get('name_short', f'Constitution {year}'),
        "source_type": raw_data.get('source_type', 'unknown'),
        "preamble": preamble,
        "summary": {
            "total_chapters": len(chapters),
            "total_sections": total_sections,
            "total_chars": len(clean_text),
        },
        "chapters": [{"chapter_number": c.chapter_number, "chapter_title": c.chapter_title, "sections": c.sections} for c in chapters]
    }

    # บันทึกไฟล์
    with open(output_dir / f"structured_{year}.json", 'w', encoding='utf-8') as f:
        json.dump(structured, f, ensure_ascii=False, indent=2)
    _save_sections_csv(structured, output_dir / f"sections_{year}.csv")
    
    print(f"Processed {input_file.name} -> sections: {total_sections}")
    return structured

### CLI
---

In [ ]:
# ค้นหาไฟล์ JSON ทั้งหมดใน /data
json_files = sorted(INPUT_DIR.glob("const_*.json"))

if not json_files:
    print("ไม่พบไฟล์ JSON ใน data (ไฟล์ต้องขึ้นต้นด้วย const_)")
else:
    all_results = []
    for file_path in json_files:
        try:
            result = process_constitution_json(file_path, OUTPUT_DIR)
            all_results.append(result)
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")

    # สร้างไฟล์รวมทุกฉบับ (Combined CSV)
    if all_results:
        combined_path = OUTPUT_DIR / "all_sections_combined.csv"
        # รวบรวม rows ทั้งหมด
        all_rows = []
        for data in all_results:
            for chap in data['chapters']:
                for sec in chap['sections']:
                    all_rows.append({
                        'constitution_id': data['id'],
                        'year_th': data['year_th'],
                        'name_short': data['name_short'],
                        'chapter_number': chap['chapter_number'],
                        'section_number': sec['section_number'],
                        'section_text': sec['text']
                    })
        
        with open(combined_path, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\n✨ เสร็จสิ้น! รวมข้อมูล {len(all_rows)} มาตรา ไว้ที่ {combined_path}")

✅ Processed const_2475.json -> sections: 78
✅ Processed const_2482.json -> sections: 3
✅ Processed const_2483.json -> sections: 7
✅ Processed const_2485.json -> sections: 5
✅ Processed const_2489.json -> sections: 117
✅ Processed const_2490a.json -> sections: 112
✅ Processed const_2490b.json -> sections: 4
✅ Processed const_2491a.json -> sections: 30
✅ Processed const_2491b.json -> sections: 2
✅ Processed const_2492.json -> sections: 261
✅ Processed const_2495.json -> sections: 194
✅ Processed const_2502.json -> sections: 23
✅ Processed const_2511.json -> sections: 266
✅ Processed const_2515.json -> sections: 28
✅ Processed const_2517.json -> sections: 342
✅ Processed const_2518.json -> sections: 12
✅ Processed const_2519.json -> sections: 31
✅ Processed const_2520.json -> sections: 38
✅ Processed const_2521.json -> sections: 333
✅ Processed const_2528.json -> sections: 12
✅ Processed const_2532.json -> sections: 8
✅ Processed const_2534a.json -> sections: 41
✅ Processed const_2534b.js